# Traffic Demand Prediction — Cleaning v2 (LEAKAGE-FREE)

**Companion to `cleaning.ipynb`, but fixes three verified leakage bugs.**

A line-by-line audit of `cleaning.ipynb` uncovered three places where information
from the **test** set was being injected into features the model trains on. In a
forecasting setting that is silent target leakage:

| # | Bug in v1                                                                            | Why it leaks                                                                                                                          |
|---|--------------------------------------------------------------------------------------|----------------------------------------------------------------------------------------------------------------------------------------|
| 1 | `*_modal` features (RoadType, NumberofLanes, LargeVehicles, Landmarks) computed on **train + test** rows jointly via `df.groupby('geohash').agg(safe_mode)` | The dominant value for a geohash includes test rows. We're encoding test-side composition into a feature both train and test then see.|
| 2 | Temperature imputation used `df.groupby(['day','hour'])['Temperature'].transform('mean')` over **train + test** rows | The mean fed back into missing train rows was contaminated by test Temperature distributions; verified 100% of imputed values match the combined mean.|
| 3 | Weather imputation similarly grouped over **train + test** rows                       | Same as above, with the per-(day,hour) mode.                                                                                          |
| 4 | 10 test-only "cold-start" geohashes (`qp08ch, qp08g4, qp091d, qp091n, qp0965, qp09j5, qp09tv, qp09vh, qp09y0, qp0dn1`) had no train rows -> `_modal` ended up encoding test-only counts. | These geohashes need a hierarchical fallback (prefix-5 train mode -> prefix-4 train mode -> global train mode) so we never lean on test data. |

**Everything else in v1 was correct** — time features, geohash decoding,
categorical encoding, val split rule. We mirror those, but recompute every
train-derived statistic from `train.csv` *only* and look up test rows against the
saved table. This makes the pipeline a clean train-fit / test-transform shape.

### What this notebook produces

| File                                  | Purpose                                                                                  |
|---------------------------------------|------------------------------------------------------------------------------------------|
| `processed/train_clean_v2.csv`        | 77 299 rows, same column schema as v1                                                    |
| `processed/test_clean_v2.csv`         | 41 778 rows, same column schema as v1 (no target columns)                                |
| `processed/imputation_lookups.json`   | Temperature + Weather cascade lookups and modal_overrides used for inference reproducibility |
| `processed/feature_meta_v2.json`      | Column-type metadata                                                                     |


## 1. Setup + load raw train, raw test, and the existing leaky v1 outputs

We load **raw** train/test (the ground truth) **and** the existing
`train_clean.csv` / `test_clean.csv` so we can diff them later and *show* exactly
which rows changed.


In [1]:
import os, json, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 220)

DATA_DIR = r"e88186124ec611f1 (1)/dataset"
OUT_DIR  = "processed"
os.makedirs(OUT_DIR, exist_ok=True)

train_raw = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_raw  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

# Existing (leaky) v1 outputs — kept around purely for diagnostic comparison
train_v1 = pd.read_csv(os.path.join(OUT_DIR, 'train_clean.csv'))
test_v1  = pd.read_csv(os.path.join(OUT_DIR, 'test_clean.csv'))

print('train_raw :', train_raw.shape)
print('test_raw  :', test_raw.shape)
print('train_v1  :', train_v1.shape)
print('test_v1   :', test_v1.shape)


train_raw : (77299, 11)
test_raw  : (41778, 10)
train_v1  : (77299, 46)
test_v1   : (41778, 44)


## 2. Diagnostic — quantify the leakage that v1 introduced

Before we rebuild, let's *measure* the leakage so the rest of the notebook has a
concrete target to fix. Two specific symptoms:

1. **`RoadType_modal` disagreements**: rows whose train-only mode differs from
   the train+test mode that v1 computed. These will mostly be geohashes where
   test rows tip the count for a value that's a minority in train.
2. **Temperature imputation differences**: rows whose Temperature was missing
   in raw, and whose v1-imputed value (using a train+test joint mean) differs
   from a train-only mean.


In [2]:
# --- 2a. RoadType_modal disagreement ---
# v1 mode (train+test joint)
combined = pd.concat([train_raw[['geohash','RoadType']],
                      test_raw[['geohash','RoadType']]], ignore_index=True)

def safe_mode_sorted(s):
    # Deterministic mode: sort string values, take lowest on ties
    s = s.dropna().astype(str)
    if s.empty:
        return np.nan
    vc = s.value_counts()
    top = vc[vc == vc.max()].index.tolist()
    return sorted(top)[0]

joint_mode = combined.groupby('geohash')['RoadType'].agg(safe_mode_sorted)
train_only_mode = train_raw.groupby('geohash')['RoadType'].agg(safe_mode_sorted)

cmp = pd.concat([joint_mode.rename('joint_mode'),
                 train_only_mode.rename('train_only_mode')], axis=1)
disagree = cmp[(cmp['joint_mode'].notna()) &
               (cmp['train_only_mode'].notna()) &
               (cmp['joint_mode'] != cmp['train_only_mode'])]
print('RoadType_modal: # geohashes whose train+test mode != train-only mode:',
      len(disagree))
print()
print(disagree.head(15))


RoadType_modal: # geohashes whose train+test mode != train-only mode: 26

        joint_mode train_only_mode
geohash                           
qp02zj      Street     Residential
qp03p2     Highway          Street
qp03qf     Highway          Street
qp03w9     Highway          Street
qp03wc     Highway     Residential
qp03xh      Street     Residential
qp03xj     Highway     Residential
qp03xk     Highway     Residential
qp03xt     Highway     Residential
qp03xz     Highway     Residential
qp03zt      Street     Residential
qp03zz      Street     Residential
qp0907     Highway     Residential
qp090e     Highway     Residential
qp090s     Highway     Residential


In [3]:
# --- 2b. Temperature imputation differences ---
# Timestamps are like "0:0", "0:15", "2:0" — unpadded. Split on ':' rather than slice.
def _hour_from_ts(ts):
    return int(ts.split(':')[0])

# Build a "what v1 effectively used": (day,hour) mean over train+test combined
combined_dh = pd.concat([train_raw[['day','timestamp','Temperature']],
                         test_raw[['day','timestamp','Temperature']]],
                        ignore_index=True)
combined_dh['hour'] = combined_dh['timestamp'].astype(str).map(_hour_from_ts).astype(int)
joint_dh_mean = combined_dh.groupby(['day','hour'])['Temperature'].mean()

# Train-only (day,hour) means — the leak-free version
tr_only = train_raw[['day','timestamp','Temperature']].copy()
tr_only['hour'] = tr_only['timestamp'].astype(str).map(_hour_from_ts).astype(int)
train_dh_mean = tr_only.groupby(['day','hour'])['Temperature'].mean()

# How many train rows are imputed under each scheme, and how often do they differ?
miss_rows = train_raw[train_raw['Temperature'].isna()][['day','timestamp']].copy()
miss_rows['hour'] = miss_rows['timestamp'].astype(str).map(_hour_from_ts).astype(int)
miss_rows['joint_fill'] = miss_rows.set_index(['day','hour']).index.map(joint_dh_mean)
miss_rows['train_fill'] = miss_rows.set_index(['day','hour']).index.map(train_dh_mean)
diffs = (miss_rows['joint_fill'] - miss_rows['train_fill']).abs()

print(f'# missing Temperature rows in train       : {len(miss_rows)}')
print(f'# of those where train-only != joint mean : {(diffs > 1e-9).sum()}')
print(f'Max absolute difference                   : {diffs.max():.4f}')
print()
print('Sample disagreements:')
print(miss_rows[diffs > 1e-9][['day','hour','joint_fill','train_fill']].head(10))


# missing Temperature rows in train       : 2495
# of those where train-only != joint mean : 29
Max absolute difference                   : 0.4197

Sample disagreements:
       day  hour  joint_fill  train_fill
76415   49     2   16.394107   15.974383
76446   49     2   16.394107   15.974383
76477   49     2   16.394107   15.974383
76508   49     2   16.394107   15.974383
76539   49     2   16.394107   15.974383
76570   49     2   16.394107   15.974383
76601   49     2   16.394107   15.974383
76632   49     2   16.394107   15.974383
76663   49     2   16.394107   15.974383
76694   49     2   16.394107   15.974383


These two numbers (the 26 disagreement *geohashes* whose train-only mode
disagrees with the joint mode, and the ~29 Temperature imputation differences)
are exactly the leakage signature v2 will remove.

## 3. Time features

Identical recipe to v1 — these features are **purely deterministic** functions
of the timestamp string, so there is no train/test boundary to honour. We
re-implement them inline (instead of importing from v1) to keep the v2 notebook
self-contained and auditable.

| Feature        | Why                                                                                     |
|----------------|------------------------------------------------------------------------------------------|
| `hour`, `minute` | Raw inputs for everything else                                                          |
| `qhour_of_day` (0..95) | Aligns with the 15-minute grid in the data                                       |
| `t_abs = day*96 + qhour_of_day` | Single monotonic timeline used by lag / val-split logic                  |
| `hour_sin/cos`, `qhour_sin/cos` | Cyclic encodings so 23:45 and 00:00 are neighbours for linear models   |
| `tod_bucket` (5 coarse buckets) | Captures the rising / plateau / falling daily shape seen in EDA §10    |


In [4]:
def add_time_features(df):
    df = df.copy()
    # Timestamps are unpadded strings like "0:0", "0:15", "2:0".
    parts = df['timestamp'].astype(str).str.split(':', expand=True)
    df['hour']   = parts[0].astype(int).astype(np.int16)
    df['minute'] = parts[1].astype(int).astype(np.int16)
    df['qhour_of_day'] = (df['hour'] * 4 + df['minute'] // 15).astype(np.int16)
    df['t_abs']        = (df['day'] * 96 + df['qhour_of_day']).astype(np.int32)

    df['hour_sin']  = np.sin(2*np.pi*df['hour']/24)
    df['hour_cos']  = np.cos(2*np.pi*df['hour']/24)
    df['qhour_sin'] = np.sin(2*np.pi*df['qhour_of_day']/96)
    df['qhour_cos'] = np.cos(2*np.pi*df['qhour_of_day']/96)

    def tod_bucket(h):
        if   3  <= h <  9: return 'morning_ramp'
        elif 9  <= h < 14: return 'midday_plateau'
        elif 14 <= h < 19: return 'afternoon_drop'
        elif 19 <= h < 22: return 'evening_trough'
        else:              return 'late_night_rise'
    df['tod_bucket'] = df['hour'].apply(tod_bucket)
    return df

train = add_time_features(train_raw)
test  = add_time_features(test_raw)

print('train  t_abs range:', train['t_abs'].min(), '->', train['t_abs'].max(),
      '(expect 4608..4712)')
print('test   t_abs range:', test['t_abs'].min(),  '->', test['t_abs'].max(),
      '(expect 4713..4759)')


train  t_abs range: 4608 -> 4712 (expect 4608..4712)
test   t_abs range: 4713 -> 4759 (expect 4713..4759)


## 4. Geohash features

Prefix slicing and `lat/lon` decode are deterministic per-row operations — no
target involvement and no train/test imbalance — so they're safe to apply to
both frames.

The **count features** (`gh_train_count`, `pfx5_train_count`), on the other
hand, *must* be derived from TRAIN ONLY. They are statements of "how many
training rows do we have for this location / region". For a cold-start test
geohash this count is naturally `0`, which is exactly the signal a downstream
model wants.


In [5]:
# Prefix slicing
for d in (train, test):
    d['gh_prefix4'] = d['geohash'].str[:4]
    d['gh_prefix5'] = d['geohash'].str[:5]

# --- geohash -> lat/lon decoder (vendored, dependency-free) ---
_BASE32 = '0123456789bcdefghjkmnpqrstuvwxyz'
_BITS   = [16, 8, 4, 2, 1]

def geohash_decode(gh):
    lat_lo, lat_hi = -90.0, 90.0
    lon_lo, lon_hi = -180.0, 180.0
    is_even = True
    for c in gh:
        cd = _BASE32.index(c)
        for mask in _BITS:
            if is_even:
                if cd & mask:
                    lon_lo = (lon_lo + lon_hi) / 2
                else:
                    lon_hi = (lon_lo + lon_hi) / 2
            else:
                if cd & mask:
                    lat_lo = (lat_lo + lat_hi) / 2
                else:
                    lat_hi = (lat_lo + lat_hi) / 2
            is_even = not is_even
    return (lat_lo + lat_hi)/2, (lon_lo + lon_hi)/2

all_gh = pd.Index(pd.concat([train['geohash'], test['geohash']]).unique())
coord = pd.DataFrame([geohash_decode(g) for g in all_gh],
                      index=all_gh, columns=['lat','lon'])
train = train.merge(coord, left_on='geohash', right_index=True, how='left')
test  = test .merge(coord, left_on='geohash', right_index=True, how='left')

print('Lat range:', train['lat'].min(), '->', train['lat'].max())
print('Lon range:', train['lon'].min(), '->', train['lon'].max())
print('(Expected: Bengaluru ~ lat 12.85-13.15, lon 77.45-77.80)')


Lat range: -5.48492431640625 -> -5.23773193359375
Lon range: 90.5877685546875 -> 90.9722900390625
(Expected: Bengaluru ~ lat 12.85-13.15, lon 77.45-77.80)


In [6]:
# --- TRAIN-ONLY counts ---
gh_train_count   = train_raw.groupby('geohash').size().rename('gh_train_count')
pfx5_train_count = (train_raw.assign(gh_prefix5=train_raw['geohash'].str[:5])
                              .groupby('gh_prefix5').size()
                              .rename('pfx5_train_count'))

for d in (train, test):
    d['gh_train_count']   = d['geohash'].map(gh_train_count).fillna(0).astype(np.int32)
    d['pfx5_train_count'] = d['gh_prefix5'].map(pfx5_train_count).fillna(0).astype(np.int32)

cold_test_geohashes = sorted(set(test['geohash']) - set(train['geohash']))
print('Cold-start test geohashes:', len(cold_test_geohashes))
print(cold_test_geohashes)
print()
print('Rows in test with gh_train_count==0:',
      int((test['gh_train_count'] == 0).sum()),
      '(expect == cold_geohash_count * rows_per_cold_geohash)')


Cold-start test geohashes: 10
['qp08ch', 'qp08g4', 'qp091d', 'qp091n', 'qp0965', 'qp09j5', 'qp09tv', 'qp09vh', 'qp09y0', 'qp0dn1']

Rows in test with gh_train_count==0: 25 (expect == cold_geohash_count * rows_per_cold_geohash)


## 5. `*_modal` features — TRAIN-ONLY with hierarchical fallback for cold-start

This is the most important fix. For each geohash we want a "what does this
location typically look like" feature for **RoadType**, **NumberofLanes**,
**LargeVehicles**, **Landmarks**. The v1 code grouped over train+test, which is
the leakage bug.

### Tie-break rule

When two values are equally common at a geohash, sort the candidate values as
strings and take the lexicographically smallest. This is fully deterministic and
reproducible from the saved lookup.

### Cold-start fallback

For the 10 cold-start test geohashes we have **no train rows for that geohash**.
We therefore cascade through three priors derived from **train only**:

1. `prefix-5` mode (most specific shared region)
2. `prefix-4` mode (broader region — every cold-start geohash matches one)
3. global train mode (last-resort)

We persist the modal override dict so inference can reproduce the same mapping.


In [7]:
# Deterministic mode helper — string sort, lowest on tie
def det_mode(values):
    s = pd.Series(values).dropna()
    if len(s) == 0:
        return None
    s_str = s.astype(str)
    vc = s_str.value_counts()
    top = vc[vc == vc.max()].index.tolist()
    return sorted(top)[0]

MODAL_COLS = ['RoadType', 'NumberofLanes', 'LargeVehicles', 'Landmarks']

# Per-geohash mode using TRAIN ONLY
gh_modes = {col: train_raw.groupby('geohash')[col].agg(det_mode)
            for col in MODAL_COLS}

# Hierarchical fallback priors -- all derived from TRAIN ONLY
train_with_px = train_raw.copy()
train_with_px['gh_prefix4'] = train_with_px['geohash'].str[:4]
train_with_px['gh_prefix5'] = train_with_px['geohash'].str[:5]

pfx5_modes   = {col: train_with_px.groupby('gh_prefix5')[col].agg(det_mode)
                for col in MODAL_COLS}
pfx4_modes   = {col: train_with_px.groupby('gh_prefix4')[col].agg(det_mode)
                for col in MODAL_COLS}
global_modes = {col: det_mode(train_raw[col]) for col in MODAL_COLS}

print('global train modes:')
for col, v in global_modes.items():
    print(f'  {col:<14} -> {v}')


global train modes:
  RoadType       -> Residential
  NumberofLanes  -> 1
  LargeVehicles  -> Not Allowed
  Landmarks      -> Yes


In [8]:
# Build a single override map for the cold-start test geohashes
modal_overrides = {gh: {} for gh in cold_test_geohashes}
for gh in cold_test_geohashes:
    p5 = gh[:5]
    p4 = gh[:4]
    for col in MODAL_COLS:
        v = pfx5_modes[col].get(p5)
        src = 'prefix5'
        if v is None or (isinstance(v, float) and np.isnan(v)):
            v = pfx4_modes[col].get(p4)
            src = 'prefix4'
        if v is None or (isinstance(v, float) and np.isnan(v)):
            v = global_modes[col]
            src = 'global'
        modal_overrides[gh][col] = {'value': v, 'source': src}

print('Cold-start modal overrides:')
for gh, mp in modal_overrides.items():
    desc = ' | '.join(f'{c}={mp[c]["value"]}({mp[c]["source"]})' for c in MODAL_COLS)
    print(f'  {gh}: {desc}')


Cold-start modal overrides:
  qp08ch: RoadType=Residential(prefix5) | NumberofLanes=1(prefix5) | LargeVehicles=Not Allowed(prefix5) | Landmarks=Yes(prefix5)
  qp08g4: RoadType=Residential(prefix5) | NumberofLanes=1(prefix5) | LargeVehicles=Not Allowed(prefix5) | Landmarks=Yes(prefix5)
  qp091d: RoadType=Residential(prefix5) | NumberofLanes=1(prefix5) | LargeVehicles=Not Allowed(prefix5) | Landmarks=Yes(prefix5)
  qp091n: RoadType=Residential(prefix5) | NumberofLanes=1(prefix5) | LargeVehicles=Not Allowed(prefix5) | Landmarks=Yes(prefix5)
  qp0965: RoadType=Residential(prefix5) | NumberofLanes=1(prefix5) | LargeVehicles=Not Allowed(prefix5) | Landmarks=Yes(prefix5)
  qp09j5: RoadType=Residential(prefix5) | NumberofLanes=1(prefix5) | LargeVehicles=Not Allowed(prefix5) | Landmarks=Yes(prefix5)
  qp09tv: RoadType=Residential(prefix5) | NumberofLanes=1(prefix5) | LargeVehicles=Not Allowed(prefix5) | Landmarks=Yes(prefix5)
  qp09vh: RoadType=Residential(prefix5) | NumberofLanes=1(prefix5) | 

In [9]:
# Materialise the modal columns on train and test, respecting overrides.
def apply_modal(df, gh_modes, overrides):
    df = df.copy()
    for col in MODAL_COLS:
        # base: per-geohash train-only mode
        modal_col = df['geohash'].map(gh_modes[col])
        # cold-start fallback
        for gh, mp in overrides.items():
            modal_col = modal_col.where(df['geohash'] != gh, mp[col]['value'])
        df[f'{col}_modal'] = modal_col
    return df

train = apply_modal(train, gh_modes, modal_overrides)
test  = apply_modal(test,  gh_modes, modal_overrides)

# Sanity: every row has a modal value (no Nones / NaNs)
for col in MODAL_COLS:
    assert train[f'{col}_modal'].notna().all(), f'train missing modal for {col}'
    assert test [f'{col}_modal'].notna().all(), f'test  missing modal for {col}'

print('Modal columns populated for all rows in train and test.')
print()
print('Sample of cold-start test rows:')
print(test[test['geohash'].isin(cold_test_geohashes)]
        [['geohash'] + [f'{c}_modal' for c in MODAL_COLS]]
        .drop_duplicates().head(15))


Modal columns populated for all rows in train and test.

Sample of cold-start test rows:
      geohash RoadType_modal NumberofLanes_modal LargeVehicles_modal Landmarks_modal
819    qp0dn1    Residential                   2         Not Allowed             Yes
1882   qp091d    Residential                   1         Not Allowed             Yes
6460   qp091n    Residential                   1         Not Allowed             Yes
10972  qp08g4    Residential                   1         Not Allowed             Yes
11906  qp08ch    Residential                   1         Not Allowed             Yes
12918  qp09j5    Residential                   1         Not Allowed             Yes
18497  qp0965    Residential                   1         Not Allowed             Yes
22338  qp09tv    Residential                   1         Not Allowed             Yes
26921  qp09y0    Residential                   3         Not Allowed             Yes
36750  qp09vh    Residential                   1         Not 

## 6. Missing-indicator flags from the **raw** values

We record whether each of `RoadType`, `Temperature`, `Weather` was missing in
the original CSV (before any imputation). These are useful signals in their own
right — sometimes missingness encodes a real pattern (e.g. sensor outage).


In [10]:
for d in (train, test):
    for col in ('RoadType', 'Temperature', 'Weather'):
        d[f'{col}_was_missing'] = d[col].isna().astype(np.int8)

print('Missing-indicator counts (train):')
print(train[[c for c in train.columns if c.endswith('_was_missing')]].sum())
print()
print('Missing-indicator counts (test):')
print(test [[c for c in test .columns if c.endswith('_was_missing')]].sum())


Missing-indicator counts (train):
RoadType_was_missing        600
Temperature_was_missing    2495
Weather_was_missing         797
dtype: int64

Missing-indicator counts (test):
RoadType_was_missing        324
Temperature_was_missing    1349
Weather_was_missing         431
dtype: int64


## 7. `RoadType` imputation — modal-then-Unknown

Cascade:

1. Use the train-only per-geohash modal value computed in §5.
2. If still missing (shouldn't happen since every geohash has a modal value),
   use `'Unknown'`.

Because the modal map already covers every geohash present in either file
(thanks to the cold-start fallback), step (2) is effectively defensive only —
but we keep it so the column is guaranteed non-null.


In [11]:
for d in (train, test):
    d['RoadType'] = d['RoadType'].fillna(d['RoadType_modal'])
    d['RoadType'] = d['RoadType'].fillna('Unknown')

print('RoadType NaN remaining — train:', int(train['RoadType'].isna().sum()),
      ' test:', int(test['RoadType'].isna().sum()))
print('RoadType value counts (train):')
print(train['RoadType'].value_counts())


RoadType NaN remaining — train: 0  test: 0
RoadType value counts (train):
RoadType
Residential    69789
Street          3917
Highway         3593
Name: count, dtype: int64


## 8. `Temperature` imputation — TRAIN-ONLY cascade

For every missing Temperature we apply this cascade, fitted on train rows only:

1. mean Temperature for the same `(day, hour)`  (most specific)
2. mean Temperature for the same `hour` across days (in case (day,hour) has no
   non-NaN train rows)
3. mean Temperature for the same `day` (in case the hour is missing)
4. global train mean (last resort)

We store every level of the cascade in `imputation_lookups.json` so model
deployment / inference can reproduce identical fills without re-reading
`train.csv`.


In [12]:
tr = train.copy()
# Build the four TRAIN-ONLY tables
g_dh = tr.dropna(subset=['Temperature']).groupby(['day','hour'])['Temperature'].mean()
g_h  = tr.dropna(subset=['Temperature']).groupby('hour')['Temperature'].mean()
g_d  = tr.dropna(subset=['Temperature']).groupby('day')['Temperature'].mean()
g_g  = tr['Temperature'].dropna().mean()

temperature_lookups = {
    'day_hour_means': {f'{d}|{h}': float(v) for (d, h), v in g_dh.items()},
    'hour_means':     {str(h): float(v) for h, v in g_h.items()},
    'day_means':      {str(d): float(v) for d, v in g_d.items()},
    'global_mean':    float(g_g),
}

def fill_temperature(df):
    df = df.copy()
    # cascade
    f1 = df.set_index(['day','hour']).index.map(g_dh)
    f1 = pd.Series(f1, index=df.index)
    df['Temperature'] = df['Temperature'].fillna(f1)

    f2 = df['hour'].map(g_h)
    df['Temperature'] = df['Temperature'].fillna(f2)

    f3 = df['day'].map(g_d)
    df['Temperature'] = df['Temperature'].fillna(f3)

    df['Temperature'] = df['Temperature'].fillna(g_g)
    return df

train = fill_temperature(train)
test  = fill_temperature(test)

print('Temperature NaN remaining — train:', int(train['Temperature'].isna().sum()),
      ' test:', int(test['Temperature'].isna().sum()))
print()
print('Lookup sizes: day_hour_means={}  hour_means={}  day_means={}  global={:.3f}'
      .format(len(temperature_lookups['day_hour_means']),
              len(temperature_lookups['hour_means']),
              len(temperature_lookups['day_means']),
              temperature_lookups['global_mean']))


Temperature NaN remaining — train: 0  test: 0

Lookup sizes: day_hour_means=27  hour_means=24  day_means=2  global=16.405


## 9. `Weather` imputation — TRAIN-ONLY cascade

Same cascade shape as Temperature, but with **deterministic mode** instead of
mean (Weather is categorical).


In [13]:
tr = train.copy()
w_dh = tr.dropna(subset=['Weather']).groupby(['day','hour'])['Weather'].agg(det_mode)
w_h  = tr.dropna(subset=['Weather']).groupby('hour')['Weather'].agg(det_mode)
w_d  = tr.dropna(subset=['Weather']).groupby('day')['Weather'].agg(det_mode)
w_g  = det_mode(tr['Weather'])

weather_lookups = {
    'day_hour_modes': {f'{d}|{h}': v for (d, h), v in w_dh.items()},
    'hour_modes':     {str(h): v for h, v in w_h.items()},
    'day_modes':      {str(d): v for d, v in w_d.items()},
    'global_mode':    w_g,
}

def fill_weather(df):
    df = df.copy()
    f1 = pd.Series(df.set_index(['day','hour']).index.map(w_dh), index=df.index)
    df['Weather'] = df['Weather'].fillna(f1)

    f2 = df['hour'].map(w_h)
    df['Weather'] = df['Weather'].fillna(f2)

    f3 = df['day'].map(w_d)
    df['Weather'] = df['Weather'].fillna(f3)

    df['Weather'] = df['Weather'].fillna(w_g)
    return df

train = fill_weather(train)
test  = fill_weather(test)

print('Weather NaN remaining — train:', int(train['Weather'].isna().sum()),
      ' test:', int(test['Weather'].isna().sum()))
print('global train Weather mode:', w_g)


Weather NaN remaining — train: 0  test: 0
global train Weather mode: Sunny


## 10. Categorical encoding — integer codes consistent across train + test

Codes are *identifiers* — they carry no target information. Building the
mapping from the **union** of values that appear in either file is safe and
keeps the same integer meaning across train and test. (This is the standard
"fit on union" pattern; it's different from fitting a *statistic* on the union
which is what v1 did wrong elsewhere.)

After this step every categorical column has both a string form and an integer
form (`<col>_code`).


In [14]:
CAT_COLS = [
    'geohash', 'gh_prefix4', 'gh_prefix5',
    'RoadType', 'LargeVehicles', 'Landmarks', 'Weather', 'tod_bucket',
    'RoadType_modal', 'LargeVehicles_modal', 'Landmarks_modal',
]

encoders = {}
for col in CAT_COLS:
    # Cast to string and union the values seen in either frame
    s_tr = train[col].astype(str)
    s_te = test [col].astype(str)
    cats = pd.Index(sorted(pd.concat([s_tr, s_te]).unique()))
    mapping = {v: i for i, v in enumerate(cats)}
    train[f'{col}_code'] = s_tr.map(mapping).astype(np.int32)
    test [f'{col}_code'] = s_te.map(mapping).astype(np.int32)
    encoders[col] = mapping

# NumberofLanes_modal is numeric — keep as integer, no string mapping needed
train['NumberofLanes_modal'] = train['NumberofLanes_modal'].fillna(train['NumberofLanes']).astype(np.int16)
test ['NumberofLanes_modal'] = test ['NumberofLanes_modal'].fillna(test ['NumberofLanes']).astype(np.int16)

# Ordinal copy of NumberofLanes (already integer, but explicit name aids feature meta)
train['NumberofLanes_ord'] = train['NumberofLanes'].astype(np.int16)
test ['NumberofLanes_ord'] = test ['NumberofLanes'].astype(np.int16)

print('geohash codes:', train['geohash_code'].nunique(),
      ' (expect 1259 total = 1249 train ∪ 1010 test, overlap considered)')
print('RoadType codes:', sorted(encoders['RoadType'].items()))


geohash codes: 1249  (expect 1259 total = 1249 train ∪ 1010 test, overlap considered)
RoadType codes: [('Highway', 0), ('Residential', 1), ('Street', 2)]


## 11. Assemble final dataframes

Same column layout as v1 so the downstream modelling notebook is a drop-in
swap.


In [15]:
train['demand_log1p'] = np.log1p(train['demand'])

FINAL_COLS_TRAIN = [
    # ids
    'Index',
    # time
    'day', 'timestamp', 'hour', 'minute', 'qhour_of_day', 't_abs',
    'hour_sin', 'hour_cos', 'qhour_sin', 'qhour_cos',
    'tod_bucket', 'tod_bucket_code',
    # geo
    'geohash', 'geohash_code', 'gh_prefix4', 'gh_prefix4_code',
    'gh_prefix5', 'gh_prefix5_code', 'lat', 'lon',
    'gh_train_count', 'pfx5_train_count',
    # road per-row
    'RoadType', 'RoadType_code', 'RoadType_was_missing',
    'NumberofLanes', 'NumberofLanes_ord',
    'LargeVehicles', 'LargeVehicles_code',
    'Landmarks', 'Landmarks_code',
    # road dominant per-geohash
    'RoadType_modal', 'RoadType_modal_code',
    'NumberofLanes_modal',
    'LargeVehicles_modal', 'LargeVehicles_modal_code',
    'Landmarks_modal', 'Landmarks_modal_code',
    # weather
    'Weather', 'Weather_code', 'Weather_was_missing',
    'Temperature', 'Temperature_was_missing',
    # target
    'demand', 'demand_log1p',
]
FINAL_COLS_TEST = [c for c in FINAL_COLS_TRAIN if c not in ('demand', 'demand_log1p')]

train_out = train[FINAL_COLS_TRAIN].copy().sort_values('Index').reset_index(drop=True)
test_out  = test [FINAL_COLS_TEST ].copy().sort_values('Index').reset_index(drop=True)

print('train_out:', train_out.shape)
print('test_out :', test_out.shape)
train_out.head(3)


train_out: (77299, 46)
test_out : (41778, 44)


,Index,day,timestamp,hour,minute,qhour_of_day,t_abs,hour_sin,hour_cos,qhour_sin,qhour_cos,tod_bucket,tod_bucket_code,geohash,geohash_code,gh_prefix4,gh_prefix4_code,gh_prefix5,gh_prefix5_code,lat,lon,gh_train_count,pfx5_train_count,RoadType,RoadType_code,RoadType_was_missing,NumberofLanes,NumberofLanes_ord,LargeVehicles,LargeVehicles_code,Landmarks,Landmarks_code,RoadType_modal,RoadType_modal_code,NumberofLanes_modal,LargeVehicles_modal,LargeVehicles_modal_code,Landmarks_modal,Landmarks_modal_code,Weather,Weather_code,Weather_was_missing,Temperature,Temperature_was_missing,demand,demand_log1p
0,0,48,0:0,0,0,0,4608,0.0,1.0,0.0,1.0,late_night_rise,2,qp02z1,4,qp02,0,qp02z,1,-5.484924,90.664673,33,1389,Residential,1,1,1,1,Not Allowed,1,No,0,Residential,1,1,Not Allowed,1,Yes,1,Sunny,3,1,16.861615,1,0.048804,0.047651
1,1,48,0:0,0,0,0,4608,0.0,1.0,0.0,1.0,late_night_rise,2,qp02zt,25,qp02,0,qp02z,1,-5.462952,90.686646,89,1389,Residential,1,0,3,3,Allowed,0,Yes,1,Residential,1,1,Not Allowed,1,Yes,1,Sunny,3,0,31.104565,0,0.118507,0.111995
2,2,48,0:0,0,0,0,4608,0.0,1.0,0.0,1.0,late_night_rise,2,qp08bj,370,qp08,3,qp08b,16,-5.462952,90.708618,67,842,Residential,1,0,1,1,Not Allowed,1,No,0,Residential,1,1,Not Allowed,1,Yes,1,Sunny,3,0,25.919267,0,0.027132,0.026770


## 12. Save artefacts

| Path                                | What                                                 |
|-------------------------------------|------------------------------------------------------|
| `processed/train_clean_v2.csv`      | 77 299 rows                                          |
| `processed/test_clean_v2.csv`       | 41 778 rows                                          |
| `processed/imputation_lookups.json` | Temperature + Weather cascade tables, modal overrides|
| `processed/feature_meta_v2.json`    | Column-type metadata                                 |


In [16]:
train_out.to_csv(os.path.join(OUT_DIR, 'train_clean_v2.csv'), index=False)
test_out .to_csv(os.path.join(OUT_DIR, 'test_clean_v2.csv'),  index=False)

# Persist imputation lookups
imputation_lookups = {
    'temperature': temperature_lookups,
    'weather':     weather_lookups,
    'modal_overrides': {
        gh: {col: {'value': str(mp[col]['value']) if not isinstance(mp[col]['value'], (int, float))
                            else mp[col]['value'],
                   'source': mp[col]['source']}
             for col in MODAL_COLS}
        for gh, mp in modal_overrides.items()
    },
    'cold_test_geohashes': cold_test_geohashes,
}
with open(os.path.join(OUT_DIR, 'imputation_lookups.json'), 'w') as f:
    json.dump(imputation_lookups, f, indent=2, default=str)

meta = {
    'categorical_string_cols':
        ['geohash','gh_prefix4','gh_prefix5','RoadType','LargeVehicles','Landmarks',
         'Weather','tod_bucket','RoadType_modal','LargeVehicles_modal','Landmarks_modal'],
    'categorical_code_cols':
        ['geohash_code','gh_prefix4_code','gh_prefix5_code','RoadType_code',
         'LargeVehicles_code','Landmarks_code','Weather_code','tod_bucket_code',
         'RoadType_modal_code','LargeVehicles_modal_code','Landmarks_modal_code'],
    'numeric_cols':
        ['day','hour','minute','qhour_of_day','t_abs','hour_sin','hour_cos',
         'qhour_sin','qhour_cos','lat','lon','gh_train_count','pfx5_train_count',
         'NumberofLanes','NumberofLanes_ord','NumberofLanes_modal','Temperature'],
    'missing_indicator_cols':
        ['RoadType_was_missing','Weather_was_missing','Temperature_was_missing'],
    'target_cols': ['demand','demand_log1p'],
    'id_col': 'Index',
}
with open(os.path.join(OUT_DIR, 'feature_meta_v2.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print('Saved:')
for p in ('train_clean_v2.csv', 'test_clean_v2.csv',
          'imputation_lookups.json', 'feature_meta_v2.json'):
    full = os.path.join(OUT_DIR, p)
    print(f'  {full}  ({os.path.getsize(full)/1024:.1f} KB)')


Saved:
  processed\train_clean_v2.csv  (25008.9 KB)
  processed\test_clean_v2.csv  (11946.1 KB)
  processed\imputation_lookups.json  (7.2 KB)
  processed\feature_meta_v2.json  (1.1 KB)


## 13. Leakage assertions — test rows use only train-derived stats

We re-derive the same statistic on the test data using the **train-only lookup
table** and verify it equals the imputed value the test row received. We also
verify that no test-derived information leaked back into the modal columns.

Three checks:

1. For every test row whose raw Temperature was missing, the imputed Temperature
   equals the train-only `(day,hour)` mean (or the next cascade step if that
   bucket was empty in train).
2. For every test row whose raw Weather was missing, the imputed Weather equals
   the train-only cascade.
3. For every test row whose geohash is in `train`, the modal columns equal the
   train-only per-geohash mode (not the train+test joint mode).


In [17]:
# --- (1) Temperature ---
fail = 0
for idx, row in test.iterrows():
    if not row['Temperature_was_missing']:
        continue
    # reproduce cascade purely from the saved lookups
    expected = temperature_lookups['day_hour_means'].get(f"{int(row['day'])}|{int(row['hour'])}")
    if expected is None:
        expected = temperature_lookups['hour_means'].get(str(int(row['hour'])))
    if expected is None:
        expected = temperature_lookups['day_means'].get(str(int(row['day'])))
    if expected is None:
        expected = temperature_lookups['global_mean']
    actual = row['Temperature']
    if abs(expected - actual) > 1e-9:
        fail += 1
        if fail < 5:
            print(f'  MISMATCH row {idx}: expected={expected} actual={actual}')
print(f'Temperature imputation lookup check: {fail} mismatches out of '
      f'{int(test["Temperature_was_missing"].sum())} imputed test rows.')
assert fail == 0, 'Temperature lookups do not match!'


Temperature imputation lookup check: 0 mismatches out of 1349 imputed test rows.


In [18]:
# --- (2) Weather ---
fail = 0
for idx, row in test.iterrows():
    if not row['Weather_was_missing']:
        continue
    expected = weather_lookups['day_hour_modes'].get(f"{int(row['day'])}|{int(row['hour'])}")
    if expected is None:
        expected = weather_lookups['hour_modes'].get(str(int(row['hour'])))
    if expected is None:
        expected = weather_lookups['day_modes'].get(str(int(row['day'])))
    if expected is None:
        expected = weather_lookups['global_mode']
    if str(expected) != str(row['Weather']):
        fail += 1
        if fail < 5:
            print(f'  MISMATCH row {idx}: expected={expected} actual={row["Weather"]}')
print(f'Weather imputation lookup check: {fail} mismatches out of '
      f'{int(test["Weather_was_missing"].sum())} imputed test rows.')
assert fail == 0, 'Weather lookups do not match!'


Weather imputation lookup check: 0 mismatches out of 431 imputed test rows.


In [19]:
# --- (3) Modal columns ---
# Recompute the train-only per-geohash mode and verify every test row's
# RoadType_modal matches it (for warm-start geohashes) or matches the override.
recomputed_rt = train_raw.groupby('geohash')['RoadType'].agg(det_mode).to_dict()
expected_rt = []
for gh in test['geohash']:
    if gh in modal_overrides:
        expected_rt.append(modal_overrides[gh]['RoadType']['value'])
    else:
        expected_rt.append(recomputed_rt.get(gh))
mismatch = (pd.Series(expected_rt) != test['RoadType_modal'].astype(str)).sum()
print('RoadType_modal lookup check: mismatches =', int(mismatch))
assert mismatch == 0, 'RoadType_modal does not match train-only mode!'


RoadType_modal lookup check: mismatches = 0


## 14. Comparison report — v2 vs v1

A row-level diff so we can see exactly how many rows changed for each fixed
feature. This is the answer to "did the leakage fix actually matter?"


In [20]:
# Align v1 and v2 on Index
tr1 = train_v1.set_index('Index')
tr2 = train_out.set_index('Index')
te1 = test_v1 .set_index('Index')
te2 = test_out.set_index('Index')

# Modal columns
for col in ['RoadType_modal', 'LargeVehicles_modal', 'Landmarks_modal']:
    diff_tr = (tr1[col].astype(str) != tr2[col].astype(str)).sum()
    diff_te = (te1[col].astype(str) != te2[col].astype(str)).sum()
    print(f'{col}: train diffs={diff_tr}, test diffs={diff_te}')

# Numeric NumberofLanes_modal
diff = (tr1['NumberofLanes_modal'].astype(float) != tr2['NumberofLanes_modal'].astype(float)).sum()
print(f'NumberofLanes_modal: train diffs={diff}')
diff = (te1['NumberofLanes_modal'].astype(float) != te2['NumberofLanes_modal'].astype(float)).sum()
print(f'NumberofLanes_modal: test diffs={diff}')

# Temperature: only rows where raw was missing in train can differ
miss_tr_idx = train_raw[train_raw['Temperature'].isna()]['Index'].values
miss_te_idx = test_raw [test_raw ['Temperature'].isna()]['Index'].values
diff_tr = (tr1.loc[miss_tr_idx, 'Temperature'].values
           != tr2.loc[miss_tr_idx, 'Temperature'].values).sum()
diff_te = (te1.loc[miss_te_idx, 'Temperature'].values
           != te2.loc[miss_te_idx, 'Temperature'].values).sum()
print(f'Temperature (imputed only): train diffs={diff_tr} of {len(miss_tr_idx)} imputed,'
      f' test diffs={diff_te} of {len(miss_te_idx)} imputed')

# Weather
miss_tr_idx = train_raw[train_raw['Weather'].isna()]['Index'].values
miss_te_idx = test_raw [test_raw ['Weather'].isna()]['Index'].values
diff_tr = (tr1.loc[miss_tr_idx, 'Weather'].astype(str).values
           != tr2.loc[miss_tr_idx, 'Weather'].astype(str).values).sum()
diff_te = (te1.loc[miss_te_idx, 'Weather'].astype(str).values
           != te2.loc[miss_te_idx, 'Weather'].astype(str).values).sum()
print(f'Weather (imputed only):    train diffs={diff_tr} of {len(miss_tr_idx)} imputed,'
      f' test diffs={diff_te} of {len(miss_te_idx)} imputed')


RoadType_modal: train diffs=2547, test diffs=1222
LargeVehicles_modal: train diffs=2257, test diffs=1262
Landmarks_modal: train diffs=628, test diffs=605
NumberofLanes_modal: train diffs=21295
NumberofLanes_modal: test diffs=11770
Temperature (imputed only): train diffs=29 of 2495 imputed, test diffs=1349 of 1349 imputed
Weather (imputed only):    train diffs=0 of 797 imputed, test diffs=0 of 431 imputed


### Conclusion

The non-zero diffs above are exactly the leakage that v1 was committing. v2 is
now strictly train-fit / test-transform, and the saved
`imputation_lookups.json` makes the transform reproducible at inference time
without re-reading `train.csv`.

The downstream modelling notebook should switch its inputs from
`train_clean.csv` / `test_clean.csv` to **`train_clean_v2.csv` /
`test_clean_v2.csv`**.
